# 02 — Department classifier

**Inputs:** `data/processed/{train,val,test}.csv` (from notebook 01)

**Outputs:** `models/artifacts/department_clf.joblib` + sidecar `department_clf.json`
(`{model_version, trained_at, classes, macro_f1, threshold}`)

**Acceptance target:** macro-F1 >= 0.85 on the held-out test split.

In [1]:
import os, sys
from pathlib import Path

# Notebook hygiene: always run from the v1 root, never hard-code absolute d:\... paths.
_here = Path.cwd()
_v1_root = _here if (_here / "pyproject.toml").exists() else _here.parents[0]
os.chdir(_v1_root)
sys.path.insert(0, str(_v1_root))

# Notebooks that touch libs.platform.registry/config must pin a profile explicitly — the
# process env has no APP_PROFILE by default, which silently falls back to "cpu" (real Ollama
# calls, no stub) and produces confusing failures/latency. Pin stub here; override the env var
# before importing libs.platform.config in the rare notebook that needs otherwise.
os.environ.setdefault("APP_PROFILE", "stub")
print("cwd:", Path.cwd(), "| APP_PROFILE:", os.environ["APP_PROFILE"])

cwd: D:\DSEP22\v1 | APP_PROFILE: stub


In [2]:
import json
from datetime import datetime, timezone

import joblib
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

RANDOM_STATE = 22

train_df = pd.read_csv("data/processed/train.csv")
val_df = pd.read_csv("data/processed/val.csv")
test_df = pd.read_csv("data/processed/test.csv")
len(train_df), len(val_df), len(test_df)

(21825, 2728, 2729)

## Features

Word 1-2 grams + char 3-5 grams (the char n-grams help with typos and the templated,
slightly-repetitive phrasing in both data sources).

In [3]:
def make_features():
    word_vec = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)
    return FeatureUnion([("word", word_vec), ("char", char_vec)])

## Baseline A: calibrated LinearSVC

In [4]:
svc_pipe = Pipeline([
    ("features", make_features()),
    ("clf", CalibratedClassifierCV(LinearSVC(class_weight="balanced", C=1.0, random_state=RANDOM_STATE), method="sigmoid", cv=3)),
])
svc_pipe.fit(train_df["text"], train_df["department"])
svc_val_pred = svc_pipe.predict(val_df["text"])
svc_macro_f1 = f1_score(val_df["department"], svc_val_pred, average="macro")
print("LinearSVC macro-F1 (val):", svc_macro_f1)
print(classification_report(val_df["department"], svc_val_pred))

LinearSVC macro-F1 (val): 1.0
                    precision    recall  f1-score   support

           billing       1.00      1.00      1.00       699
     field_service       1.00      1.00      1.00        16
           general       1.00      1.00      1.00      1195
network_operations       1.00      1.00      1.00        25
         retention       1.00      1.00      1.00        95
 technical_support       1.00      1.00      1.00       698

          accuracy                           1.00      2728
         macro avg       1.00      1.00      1.00      2728
      weighted avg       1.00      1.00      1.00      2728



## Baseline B: Logistic Regression

In [5]:
lr_pipe = Pipeline([
    ("features", make_features()),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
])
lr_pipe.fit(train_df["text"], train_df["department"])
lr_val_pred = lr_pipe.predict(val_df["text"])
lr_macro_f1 = f1_score(val_df["department"], lr_val_pred, average="macro")
print("LogisticRegression macro-F1 (val):", lr_macro_f1)
print(classification_report(val_df["department"], lr_val_pred))

LogisticRegression macro-F1 (val): 1.0
                    precision    recall  f1-score   support

           billing       1.00      1.00      1.00       699
     field_service       1.00      1.00      1.00        16
           general       1.00      1.00      1.00      1195
network_operations       1.00      1.00      1.00        25
         retention       1.00      1.00      1.00        95
 technical_support       1.00      1.00      1.00       698

          accuracy                           1.00      2728
         macro avg       1.00      1.00      1.00      2728
      weighted avg       1.00      1.00      1.00      2728



## Calibration check

`ClassifierPort` must return usable confidences — the triage priority threshold and the
sklearn->LLM->rules classifier fallback order both depend on them being meaningful, not just a
tie-breaking score.

In [6]:
import numpy as np

winner_pipe = svc_pipe if svc_macro_f1 >= lr_macro_f1 else lr_pipe
winner_name = "LinearSVC (calibrated)" if svc_macro_f1 >= lr_macro_f1 else "LogisticRegression"
print("winner:", winner_name)

proba = winner_pipe.predict_proba(val_df["text"])
top1_conf = proba.max(axis=1)
top1_correct = (winner_pipe.classes_[proba.argmax(axis=1)] == val_df["department"].values)

bins = np.linspace(0, 1, 11)
bin_ids = np.digitize(top1_conf, bins) - 1
for b in sorted(set(bin_ids)):
    mask = bin_ids == b
    if mask.sum() == 0:
        continue
    print(f"confidence [{bins[b]:.1f},{bins[b+1] if b+1 < len(bins) else 1.0:.1f}): "
          f"n={mask.sum():4d} accuracy={top1_correct[mask].mean():.3f} mean_conf={top1_conf[mask].mean():.3f}")

winner: LinearSVC (calibrated)
confidence [0.9,1.0): n=2728 accuracy=1.000 mean_conf=0.999


## Final evaluation on the held-out test split

In [7]:
test_pred = winner_pipe.predict(test_df["text"])
test_macro_f1 = f1_score(test_df["department"], test_pred, average="macro")
print("held-out test macro-F1:", test_macro_f1)
print(classification_report(test_df["department"], test_pred))
assert test_macro_f1 >= 0.85, f"macro-F1 {test_macro_f1:.3f} is below the 0.85 acceptance target"

held-out test macro-F1: 1.0
                    precision    recall  f1-score   support

           billing       1.00      1.00      1.00       699
     field_service       1.00      1.00      1.00        16
           general       1.00      1.00      1.00      1195
network_operations       1.00      1.00      1.00        25
         retention       1.00      1.00      1.00        95
 technical_support       1.00      1.00      1.00       699

          accuracy                           1.00      2729
         macro avg       1.00      1.00      1.00      2729
      weighted avg       1.00      1.00      1.00      2729



In [8]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(test_df["department"], test_pred, ax=ax, xticks_rotation=45)
fig.tight_layout()
report_dir = Path("evaluation/reports")
report_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(report_dir / "department_classifier_confusion_matrix.png", dpi=120)
print("saved confusion matrix plot")

saved confusion matrix plot


## Save the artifact

In [9]:
artifacts_dir = Path("models/artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

model_version = f"department_clf-{datetime.now(timezone.utc):%Y%m%d%H%M%S}"
joblib.dump({"pipeline": winner_pipe}, artifacts_dir / "department_clf.joblib")

sidecar = {
    "model_version": model_version,
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "classes": sorted(winner_pipe.classes_.tolist()),
    "macro_f1": test_macro_f1,
    "threshold": 0.5,
    "winner": winner_name,
}
(artifacts_dir / "department_clf.json").write_text(json.dumps(sidecar, indent=2), encoding="utf-8")
sidecar

{'model_version': 'department_clf-20260809112402',
 'trained_at': '2026-08-09T11:24:02.451100+00:00',
 'classes': ['billing',
  'field_service',
  'general',
  'network_operations',
  'retention',
  'technical_support'],
 'macro_f1': 1.0,
 'threshold': 0.5,
 'winner': 'LinearSVC (calibrated)'}

In [10]:
with open("evaluation/reports/metrics.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps({
        "notebook": "02_department_classifier",
        "winner": winner_name,
        "val_macro_f1": svc_macro_f1 if winner_name.startswith("LinearSVC") else lr_macro_f1,
        "test_macro_f1": test_macro_f1,
    }) + "\n")
print("appended metrics row")

appended metrics row
